In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Model_Credit") \
    .getOrCreate()

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [9]:
spark

In [10]:
df_test = spark.read.csv("C:/Users/celso/repos/Model_Credit/data/application_test.csv", header=True, inferSchema=True)
df_train = spark.read.csv("C:/Users/celso/repos/Model_Credit/data/application_train.csv", header=True, inferSchema=True)
df_bureau_balance = spark.read.csv("C:/Users/celso/repos/Model_Credit/data/bureau_balance.csv", header=True, inferSchema=True)
df_bureau = spark.read.csv("C:/Users/celso/repos/Model_Credit/data/bureau.csv", header=True, inferSchema=True)
df_credit_card_balance = spark.read.csv("C:/Users/celso/repos/Model_Credit/data/credit_card_balance.csv", header=True, inferSchema=True)
df_credit_description = spark.read.csv("C:/Users/celso/repos/Model_Credit/data/HomeCredit_columns_description.csv", header=True, inferSchema=True)
df_payments = spark.read.csv("C:/Users/celso/repos/Model_Credit/data/installments_payments.csv", header=True, inferSchema=True)
df_cash_balance = spark.read.csv("C:/Users/celso/repos/Model_Credit/data/POS_CASH_balance.csv", header=True, inferSchema=True)
df_previous_application = spark.read.csv("C:/Users/celso/repos/Model_Credit/data/previous_application.csv", header=True, inferSchema=True)
df_sample_submission = spark.read.csv("C:/Users/celso/repos/Model_Credit/data/sample_submission.csv", header=True, inferSchema=True)

In [11]:
df_credit_description.show(219,truncate=False)

+---+----------------------------+----------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------+
|_c0|Table                       |Row                         |Description                                                                                                                                                                                                                                                                         |Special                              |
+---+----------------------------+----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
df_train.show(truncate=False)

+----------+------+------------------+-----------+------------+---------------+------------+----------------+----------+-----------+---------------+---------------+--------------------+-----------------------------+--------------------+-----------------+--------------------------+----------+-------------+-----------------+---------------+-----------+----------+--------------+---------------+----------------+----------+----------+---------------+---------------+--------------------+---------------------------+--------------------------+-----------------------+--------------------------+--------------------------+---------------------------+----------------------+----------------------+-----------------------+----------------------+-------------------+-------------------+-------------------+--------------+----------------+---------------------------+------------------+--------------+-------------+-------------+-------------+-------------+------------+--------------------+--------------+-

In [13]:
print(df_train.columns)
print(df_test.columns)

['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELE

In [14]:
# df_train = df_tran.filter(F.col(''))
df_train.show(truncate=False)

+----------+------+------------------+-----------+------------+---------------+------------+----------------+----------+-----------+---------------+---------------+--------------------+-----------------------------+--------------------+-----------------+--------------------------+----------+-------------+-----------------+---------------+-----------+----------+--------------+---------------+----------------+----------+----------+---------------+---------------+--------------------+---------------------------+--------------------------+-----------------------+--------------------------+--------------------------+---------------------------+----------------------+----------------------+-----------------------+----------------------+-------------------+-------------------+-------------------+--------------+----------------+---------------------------+------------------+--------------+-------------+-------------+-------------+-------------+------------+--------------------+--------------+-

In [15]:
df_train.groupBy('NAME_CONTRACT_TYPE','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE','OCCUPATION_TYPE','REGION_RATING_CLIENT').count().show()

+------------------+--------------------+--------------------+--------------------+--------------------+-----+
|NAME_CONTRACT_TYPE|    NAME_INCOME_TYPE| NAME_EDUCATION_TYPE|     OCCUPATION_TYPE|REGION_RATING_CLIENT|count|
+------------------+--------------------+--------------------+--------------------+--------------------+-----+
|   Revolving loans|             Working|Secondary / secon...|High skill tech s...|                   3|   55|
|   Revolving loans|             Working|    Higher education|Waiters/barmen staff|                   2|   14|
|        Cash loans|Commercial associate|   Incomplete higher|Private service s...|                   2|   24|
|        Cash loans|             Working|    Higher education|      Security staff|                   3|   78|
|   Revolving loans|       State servant|Secondary / secon...|            Laborers|                   1|   23|
|   Revolving loans|             Working|Secondary / secon...|             Drivers|                   2|  629|
|

In [16]:
df_payments.show()

+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+
|SK_ID_PREV|SK_ID_CURR|NUM_INSTALMENT_VERSION|NUM_INSTALMENT_NUMBER|DAYS_INSTALMENT|DAYS_ENTRY_PAYMENT|AMT_INSTALMENT|AMT_PAYMENT|
+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+
|   1054186|    161674|                   1.0|                    6|        -1180.0|           -1187.0|       6948.36|    6948.36|
|   1330831|    151639|                   0.0|                   34|        -2156.0|           -2156.0|      1716.525|   1716.525|
|   2085231|    193053|                   2.0|                    1|          -63.0|             -63.0|       25425.0|    25425.0|
|   2452527|    199697|                   1.0|                    3|        -2418.0|           -2426.0|      24350.13|   24350.13|
|   2714724|    167756|                   1.0|                    2|        -1383.0